In [3]:
import pandas as pd
from pathlib import Path
from pedotransfer_functions import PedotransferFunctionsWosten
from van_genuchten import ClassicalSoilWaterBalanceParameterProvider
from types import SimpleNamespace
import yaml


def main():
    soil_csv_fp = Path("/Users/panyue/PycharmProjects/wofost_example_test/input/soil_par.csv")
    input_dir = Path("/Users/panyue/PycharmProjects/wofost_example_test/input")
    agro_dir = input_dir / "site"
    agro_dir.mkdir(parents=True, exist_ok=True)

    CRAIRC = 0.03
    RDMSOL = 120
    theta_r = 0.01
    topSoil = True

    # load CSV
    df = pd.read_csv(soil_csv_fp)
    df.columns = df.columns.astype(str).str.strip()

    required_cols = ["ID_field", "C", "D", "S", "OM"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for _, row in df.iterrows():
        try:
            ID_field = str(row["ID_field"]).strip()
            C = float(row["C"])
            D = float(row["D"])
            S = float(row["S"])
            OM = float(row["OM"])

            # basic checks
            if C <= 0:
                raise ValueError("C must be > 0")
            if D <= 0:
                raise ValueError("D must be > 0")
            if S <= 0:
                raise ValueError("S must be > 0")
            if OM <= 0:
                OM = 0.01

            # calculate Van Genuchten parameters
            vg_dict = PedotransferFunctionsWosten(C, D, S, OM, theta_r, topSoil)
            vg = SimpleNamespace(**vg_dict)

            # calculate WOFOST soil parameters
            soil_dict = ClassicalSoilWaterBalanceParameterProvider(
                vg.alpha,
                vg.k_sat,
                vg.n,
                vg.theta_r,
                vg.theta_s,
                CRAIRC=CRAIRC,
                RDMSOL=RDMSOL
            )

            # prepare one YAML content per field
            yaml_data = {
                "ID_field": ID_field,
                "input_soil": {
                    "C": C,
                    "D": D,
                    "S": S,
                    "OM": OM,
                },
                "van_genuchten": {
                    "alpha": float(vg.alpha),
                    "k_sat": float(vg.k_sat),
                    "n": float(vg.n),
                    "theta_r": float(vg.theta_r),
                    "theta_s": float(vg.theta_s),
                },
                "wofost_soil_parameters": {
                    key: float(value) if isinstance(value, (int, float)) else value
                    for key, value in soil_dict.items()
                }
            }

            yaml_fp = agro_dir / f"site_{ID_field}.yaml"
            with open(yaml_fp, "w", encoding="utf-8") as f:
                yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

            print(f"Saved: {yaml_fp}")

        except Exception as e:
            print(f"Skipped {row.get('ID_field', 'unknown')}: {e}")


if __name__ == "__main__":
    main()

Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR01_01.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR01_02.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR01_03.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR02_01.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR02_02.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR02_03.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR03_01.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR03_02.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR03_03.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR04_01.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_test/input/site/site_DR04_02.yaml
Saved: /Users/panyue/PycharmProjects/wofost_example_te